# PFI · H04 — Auditoría de Dice/IoU sagital por clase · v1

**Objetivo:** localizar evidencia experimental del modelo sagital definitivo y, *solo si existen predicciones y máscaras correspondientes al test original*, reconstruir Dice/IoU por `vertebra_group`, `canal` y `disc_group` sin entrenar ni modificar archivos fuente.

**Procedencia verificada en GitHub (`PFI_MVPTest_Enzo_AImodule/main`):** checkpoint `sagittal_spider_multiclass_final_best.pt`; clases `0=background`, `1=vertebra_group`, `2=canal`, `3=disc_group`; split por paciente `152/33/33`, cortes `5271/1174/1218`; Dice test macro sin fondo **0.8934316063846954**, IoU **0.8079981902423006**; SHA-256 del checkpoint **`cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944`**; hash de índice de entrenamiento **`2720b7218c92870f6f0a000b57111ed36b5cf3b78c716f244f427ca7fee4a4ba`**. Fuente: `models/final/sagittal_spider_multiclass_final_best.pt.manifest.json` y model card.

**Importante:** el promedio original puede haberse calculado por corte/paciente u otro criterio. Un promedio global basado en píxeles **no se declarará equivalente** al reportado mientras no se coteje con el notebook fuente `45_gcs_spider_final_training.ipynb`. La ausencia de archivos produce **INCONCLUSO**, nunca valores inventados.

Este notebook trabaja sobre Google Drive, **sin búsqueda recursiva**, sin GPU, sin descarga de pacientes, sin ejecutar inferencia ni modificar modelos. Solo guarda salidas agregadas y anónimas en `/content` (no dentro de Drive).

In [ ]:
# 1. Montar Drive y configurar rutas (editar solamente esta celda)
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/PFI_MVP')
OUT_DIR = Path('/content/pfi_h04_auditoria_sagital')

# Carpeta(s) ADICIONALES y CONCRETAS donde podría estar el reporte original.
# No hace falta indicar carpetas genéricas con miles de archivos.
EXTRA_REPORT_DIRS = []  # Ej.: [Path('/content/drive/MyDrive/PFI_MVP/results/gcs_spider_final_training')]
# También se pueden señalar directamente reportes JSON producidos por el entrenamiento original.
EXPLICIT_REPORTS = []   # Ej.: [Path('/content/drive/MyDrive/.../gcs_spider_final_training_summary.json')]

# OPCIONAL: si tenés el checkpoint .pt materializado, indicar su ruta exacta.
# Calcularemos su SHA-256, sin cargarlo ni inferir.
CHECKPOINT_PATH = None  # Ej.: ROOT / 'models/final/sagittal_spider_multiclass_final_best.pt'
# OPCIONAL: índice CSV EXACTO usado en el entrenamiento; sirve para hash/trazabilidad.
TRAINING_INDEX_CSV = None  # Ej.: ROOT / 'results/..../spider_training_index.csv'

# MODO REEVALUACIÓN: dejar vacíos hasta confirmar que son las predicciones del MISMO test.
# Ambas carpetas deben tener archivos .npy 2D alineados, con IGUAL nombre de base.
PRED_DIR = None  # Ej.: ROOT / 'outputs/spider_test_predictions'
GT_DIR = None    # Ej.: ROOT / 'outputs/spider_test_masks_model_space'
# CSV que enumera CADA CORTE de prueba con columns configuradas abajo.
TEST_INDEX_CSV = None  # Ej.: ROOT / 'results/spider_test_slice_index.csv'
TEST_SAMPLE_COL = 'sample_id'   # stem exacto de los archivos .npy
TEST_PATIENT_COL = 'patient_id' # identificador local, nunca se exporta
TEST_SPLIT_COL = 'split'        # train / val / test
# Si las GT llevan labels crudos, PEGAR mapping exacto del checkpoint original.
# Con {} se exige que las GT ya usen 0,1,2,3. No adivinar mapping.
RAW_TO_MODEL = {}  # Ej. conceptual: {0:0, 10:1, 20:2, 30:3}
# Opcional: manifiesto que vincule las predicciones al checkpoint y al test.
PRED_PROVENANCE_JSON = None  # Ej.: ROOT / 'results/.../predictions_manifest.json'

EXPECTED = {
  'model_version':'sagittal-spider-final-v1',
  'checkpoint_sha256':'cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944',
  'training_index_sha256':'2720b7218c92870f6f0a000b57111ed36b5cf3b78c716f244f427ca7fee4a4ba',
  'patient_counts':{'train':152,'val':33,'test':33},
  'slice_counts':{'train':5271,'val':1174,'test':1218},
  'dice_macro_foreground':0.8934316063846954,
  'iou_macro_foreground':0.8079981902423006,
  'class_names':['background','vertebra_group','canal','disc_group'],
}
print('ROOT existe:', ROOT.exists())
print('Modo:', 'reevaluación opcional' if PRED_DIR is not None or GT_DIR is not None else 'búsqueda de evidencia original (rápido)')


In [ ]:
# 2. Funciones de integridad y búsqueda ACOTADA: sin ROOT.rglob ni os.walk
import csv, hashlib, json, math, re, zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
import numpy as np

def sha256_file(path, chunk=4*1024*1024):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            block=f.read(chunk)
            if not block:break
            h.update(block)
    return h.hexdigest()

def as_path(value):
    return Path(value) if value is not None else None

def immediate_subdirs(path, max_entries=450):
    path=Path(path)
    if not path.is_dir(): return []
    result=[]
    for i,p in enumerate(path.iterdir()):
        if i>=max_entries:
            print('ATENCIÓN: límite de entradas alcanzado en', path.name)
            break
        if p.is_dir(): result.append(p)
    return result

def shallow_candidate_reports(root, extra_dirs=()):
    # Inspección de profundidad máxima 2 SOLO en resultados, outputs, modelos, artefactos.
    parents=[root/'results', root/'outputs', root/'models', root/'reports', root/'artifacts']
    selected_dirs=list(parents)
    allowed_dir=re.compile(r'spider|sagit|gcs|train|eval|metric|held|test|release|qual|final|e12',re.I)
    for parent in parents:
        selected_dirs += [d for d in immediate_subdirs(parent) if allowed_dir.search(d.name)]
    selected_dirs += [Path(p) for p in extra_dirs]
    selected_dirs=list(dict.fromkeys(str(p) for p in selected_dirs))
    patterns=re.compile(r'spider|sagit|gcs|train|eval|metric|class|summary|split|test|final|quality|report|result',re.I)
    out=[]
    for directory in selected_dirs:
        d=Path(directory)
        if not d.is_dir():continue
        names=0
        for p in d.iterdir():
            names+=1
            if names>600:
                print('ATENCIÓN: directorio muy grande (inspección limitada):',d.name)
                break
            if p.is_file() and p.suffix.lower() in {'.json','.csv'} and patterns.search(p.name):
                out.append(p)
    return sorted({str(p):p for p in out}.values(),key=lambda p:str(p))

def extract_metric_candidates(obj, max_depth=8):
    indicators=re.compile(r'dice|iou|jaccard|per.?class|foreground|held.?out|checkpoint|index.?sha|mapping|class.?name|patient.?count|slice.?count',re.I)
    matches=[]
    def visit(node,keypath,depth):
        if depth>max_depth or len(matches)>160:return
        # No enviar métricas por caso/paciente individual en el reporte anonimizado.
        if re.search(r'(?:^|[.\[])(?:patients|patient|cases|case|subjects|subject|studies|study|images|image|files|file)(?:[.\[]|$)',keypath,re.I):
            return
        parent_metric=re.search(r'(?:dice|iou|jaccard|per.?class|class.?metric)',keypath,re.I)
        if isinstance(node,dict):
            for key,val in node.items():
                here=f'{keypath}.{key}' if keypath else str(key)
                selected=bool(indicators.search(str(key)) or parent_metric)
                if selected and isinstance(val,(int,float,bool,type(None),str)):
                    token=str(val)
                    if '/' in token or chr(92) in token or len(token)>140:token='[texto_omitido]'
                    matches.append((here,token))
                if isinstance(val,(dict,list)):visit(val,here,depth+1)
        elif isinstance(node,list):
            for i,v in enumerate(node[:25]):
                here=f'{keypath}[{i}]'
                if parent_metric and isinstance(v,(int,float,bool,type(None))):matches.append((here,str(v)))
                elif isinstance(v,(dict,list)):visit(v,here,depth+1)
    visit(obj,'',0)
    return matches

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Funciones listas; NO se recorrerá recursivamente Drive.')


In [ ]:
# 3. Detectar reportes originales, sin abrir datasets ni imágenes
candidate_files=shallow_candidate_reports(ROOT, EXTRA_REPORT_DIRS)
for p in EXPLICIT_REPORTS:
    p=Path(p)
    if p.is_file() and p not in candidate_files: candidate_files.append(p)
print('Candidatos superficiales:', len(candidate_files))
for p in candidate_files[:70]:
    # Sólo ruta relativa al workspace para orientar la selección; no sale al ZIP.
    try:label=str(p.relative_to(ROOT))
    except ValueError:label=p.name
    print(' •', label)
if len(candidate_files)>70:print('... listado acotado a 70; para forzar otro reporte usá EXPLICIT_REPORTS.')
print('\nSi no aparece el resumen original, buscá manualmente en Drive y agregá su ruta a EXPLICIT_REPORTS; no ejecutes una búsqueda global.')


In [ ]:
# 4. Auditar reportes JSON pequeños y hashes de fuentes declaradas
source_audit=[]
for p in candidate_files:
    if p.suffix.lower()!='.json':continue
    try:
        size=p.stat().st_size
        if size>12_000_000:
            source_audit.append({'file':p.name,'status':'saltado_mayor_12MB'});continue
        obj=json.loads(p.read_text(encoding='utf-8-sig'))
        pairs=extract_metric_candidates(obj)
        if pairs:
            source_audit.append({'file':p.name,'status':'legible','sha256':sha256_file(p),
                                 'fields':dict(pairs[:90]),'n_metric_fields':len(pairs)})
    except Exception as e:
        source_audit.append({'file':p.name,'status':'error_lectura','error_type':type(e).__name__})
print('Reportes JSON con campos relevantes:', sum(r['status']=='legible' for r in source_audit))
for record in source_audit[:30]:
    if record['status']!='legible':continue
    print('\n',record['file'], '|',record['sha256'][:12], '| campos:',record['n_metric_fields'])
    for k,v in list(record['fields'].items())[:22]: print('   ',k,':',v)

integrity={}
for key,path,expected in [('checkpoint',CHECKPOINT_PATH,EXPECTED['checkpoint_sha256']),
                         ('training_index',TRAINING_INDEX_CSV,EXPECTED['training_index_sha256'])]:
    p=as_path(path)
    if p is None:
        integrity[key]={'status':'no_proporcionado'}
    elif not p.is_file():
        integrity[key]={'status':'archivo_no_encontrado'}
    else:
        value=sha256_file(p)
        integrity[key]={'status':'match' if value.lower()==expected.lower() else 'mismatch',
                        'sha256':value,'filename':p.name}
print('\nIntegridad de fuentes declaradas:',integrity)

if not any(r['status']=='legible' for r in source_audit):
    print('NO SE ENCONTRÓ REPORTE ORIGINAL: H04 sigue abierto; configurar EXPLICIT_REPORTS.')


## Reevaluación opcional (solo si tenés las predicciones y las GT del **mismo test original**)

Este paso es independiente del anterior. Necesita: (a) carpetas `.npy` 2D de predicciones y GT ya alineadas, (b) CSV **por corte** con `sample_id`, `patient_id` y `split`, (c) mapping correcto de las etiquetas, y (d) evidencia de origen de esas predicciones. **No genera máscaras ni ejecuta inferencia**, porque eso exigiría reproducir exactamente el preprocessing y el checkpoint originales.

Una métrica recalculada aquí puede diferir del Dice 0,8934316 original por la agregación por imagen/paciente/global, la política de clases vacías o el preprocesamiento. No «ajustar» valores para hacerlos coincidir. Si los archivos no están, la celda siguiente se saltea limpiamente.

In [ ]:
# 5. Verificar pertenencia al held-out y pareamiento sin usar similitud de nombres
from pathlib import Path
pairs=[]
paired_status='no_configurado'
coverage={}
raw_test=[]
if any(x is not None for x in (PRED_DIR,GT_DIR,TEST_INDEX_CSV)):
    if not all(x is not None for x in (PRED_DIR,GT_DIR,TEST_INDEX_CSV)):
        raise ValueError('Configurar PRED_DIR, GT_DIR y TEST_INDEX_CSV juntos; no se admiten ejecuciones parciales.')
    pred_dir,gt_dir,index_path=map(Path,(PRED_DIR,GT_DIR,TEST_INDEX_CSV))
    if not (pred_dir.is_dir() and gt_dir.is_dir() and index_path.is_file()):
        raise FileNotFoundError('Una carpeta o índice de test indicado no existe.')
    with index_path.open(encoding='utf-8-sig',newline='') as f:
        reader=csv.DictReader(f)
        necessary={TEST_SAMPLE_COL,TEST_PATIENT_COL,TEST_SPLIT_COL}
        if not necessary.issubset(reader.fieldnames or []):
            raise ValueError('Índice sin las columnas necesarias: '+str(necessary-set(reader.fieldnames or [])))
        raw_test=list(reader)
    grouped=defaultdict(set)
    for row in raw_test:
        sp=(row.get(TEST_SPLIT_COL) or '').strip().lower()
        pid=(row.get(TEST_PATIENT_COL) or '').strip()
        if sp not in {'train','val','test'} or not pid:
            raise ValueError('Split o paciente vacío/inválido en índice.')
        grouped[sp].add(pid)
    if any(grouped[a]&grouped[b] for a,b in (('train','val'),('train','test'),('val','test'))):
        raise ValueError('FUGA: pacientes repetidos entre splits en el CSV suministrado.')
    test_rows=[r for r in raw_test if (r[TEST_SPLIT_COL] or '').strip().lower()=='test']
    stems=[]
    for row in test_rows:
        stem=(row.get(TEST_SAMPLE_COL) or '').strip()
        if not stem or Path(stem).name!=stem or Path(stem).suffix:
            raise ValueError('sample_id debe ser un stem simple sin directorio ni extensión.')
        stems.append(stem)
    if len(set(stems))!=len(stems):raise ValueError('Índice de test con sample_id duplicados.')
    if not test_rows:raise ValueError('Índice suministrado sin filas test.')
    missing=[]
    for row,stem in zip(test_rows,stems):
        pr=pred_dir/(stem+'.npy'); gt=gt_dir/(stem+'.npy')
        if not pr.is_file() or not gt.is_file():missing.append(stem)
        else:pairs.append((pr,gt,row[TEST_PATIENT_COL].strip()))
    all_pred={p.stem for p in pred_dir.glob('*.npy')}
    all_gt={p.stem for p in gt_dir.glob('*.npy')}
    extra_pred=all_pred-set(stems);extra_gt=all_gt-set(stems)
    coverage={'test_slices':len(test_rows),'test_patient_ids':len(grouped['test']),
              'pairs':len(pairs),'missing_pairs':len(missing),'extra_preds':len(extra_pred),'extra_gt':len(extra_gt),
              'match_test_patients_33':len(grouped['test'])==33,
              'match_test_slices_1218':len(test_rows)==1218,
              'index_sha256':sha256_file(index_path)}
    print('Cobertura:',coverage)
    if missing:raise ValueError(f'{len(missing)} pares faltantes: NO calcular métricas parciales')
    if extra_pred or extra_gt:raise ValueError('Existen archivos extra no explicados por el test_index. Aislar test de train/val.')
    paired_status='pareado_completo'
else:
    print('MODO REPORTES: sin predicciones/GT configuradas. Se omite reevaluación.')


In [ ]:
# 6. Calcular Dice/IoU por clase con 3 criterios claramente diferenciados
CLASS_NAMES=EXPECTED['class_names']
per_class=[]
recomputed={}
def read_2d_mask(path, mapping=None):
    a=np.load(path, allow_pickle=False)
    if a.ndim!=2:raise ValueError('Archivo no es máscara 2D.')
    if a.dtype.kind not in 'iub':raise ValueError('Máscara no contiene etiquetas enteras.')
    if mapping:
        outside=set(np.unique(a).tolist())-set(int(k) for k in mapping)
        if outside:raise ValueError('GT con etiquetas sin mapping; detenido.')
        b=np.zeros(a.shape,dtype=np.int32)
        for raw,cls in mapping.items():b[a==int(raw)]=int(cls)
        a=b
    values=set(int(x) for x in np.unique(a))
    if not values.issubset({0,1,2,3}):raise ValueError('Labels fuera del dominio 0,1,2,3: revisar mapping.')
    return a

def metrics_for_pairs(pairs,mapping=None):
    # Macro de suma de píxeles, promedio por corte y promedio por paciente
    total=np.zeros((4,3),dtype=np.int64)  # intersect, pred pixels, gt pixels
    per_slice=defaultdict(list)
    per_patient=defaultdict(lambda:defaultdict(list))
    n=0
    for pred_path,gt_path,pid in pairs:
        pred=read_2d_mask(pred_path)
        gt=read_2d_mask(gt_path,mapping)
        if pred.shape!=gt.shape:raise ValueError('Máscaras con geometría distinta; no redimensionar automáticamente.')
        for c in range(4):
            p=(pred==c);g=(gt==c)
            inter=int(np.count_nonzero(p&g));pp=int(np.count_nonzero(p));gg=int(np.count_nonzero(g))
            total[c]+=inter,pp,gg
            den=pp+gg; union=pp+gg-inter
            ds=2*inter/den if den else None
            io=inter/union if union else None
            per_slice[c].append((ds,io))
            per_patient[c][pid].append((ds,io))
        n+=1
        if n%100==0:print(f'Métricas calculadas: {n}/{len(pairs)}',flush=True)
    def avg_not_null(vals):
        valid=[float(x) for x in vals if x is not None]
        return float(np.mean(valid)) if valid else None
    classes=[]
    for c,name in enumerate(CLASS_NAMES):
        inter,pp,gg=map(int,total[c])
        den=pp+gg;u=pp+gg-inter
        gp=2*inter/den if den else None;gi=inter/u if u else None
        slice_dice=avg_not_null([x[0] for x in per_slice[c]])
        slice_iou=avg_not_null([x[1] for x in per_slice[c]])
        patient_dice=avg_not_null([avg_not_null([x[0] for x in vals]) for vals in per_patient[c].values()])
        patient_iou=avg_not_null([avg_not_null([x[1] for x in vals]) for vals in per_patient[c].values()])
        classes.append({'class_id':c,'name':name,'intersection':inter,'pred_pixels':pp,'gt_pixels':gg,
                        'dice_global_pixels':gp,'iou_global_pixels':gi,
                        'dice_mean_slice':slice_dice,'iou_mean_slice':slice_iou,
                        'dice_mean_patient':patient_dice,'iou_mean_patient':patient_iou,
                        'n_slices_defined':sum(x[0] is not None for x in per_slice[c]),
                        'n_patients_with_defined_score':sum(any(x[0] is not None for x in v) for v in per_patient[c].values())})
    def fg(field):return avg_not_null([entry[field] for entry in classes[1:]])
    return {'n_slices':n,'classes':classes,
        'macro_foreground':{key:fg(key) for key in ('dice_global_pixels','iou_global_pixels',
                                                   'dice_mean_slice','iou_mean_slice','dice_mean_patient','iou_mean_patient')},
        'empty_class_policy':'both-empty: excluded (None), gt-empty+prediction-nonempty: 0',
        'no_anatomical_registration_verification':True}

if paired_status=='pareado_completo':
    if any(int(v) not in (0,1,2,3) for v in RAW_TO_MODEL.values()):
        raise ValueError('Mapping invalido, class ids fuera 0..3.')
    recomputed=metrics_for_pairs(pairs,RAW_TO_MODEL)
    print('\nDice / IoU por clase (tres metodologías distintas):')
    for item in recomputed['classes']:
        print(item['class_id'],item['name'],'global=',item['dice_global_pixels'],
              'mean_slice=',item['dice_mean_slice'],'mean_patient=',item['dice_mean_patient'])
    print('\nMacros foreground recalculados:',recomputed['macro_foreground'])
    diffs={k:abs(v-EXPECTED['dice_macro_foreground']) for k,v in recomputed['macro_foreground'].items()
           if v is not None and k.startswith('dice')}
    print('Distancias respecto del macro Dice oficial (solo comparación, NO validación automática):',diffs)
else:
    print('Sin pares oficiales: NO se publican métricas por clase nuevas.')


In [ ]:
# 7. Verificar atribución experimental, generar reporte anonimizado y ZIP
source_provenance={'original_model_metrics_from_manifest':EXPECTED,
                   'provenance_of_reports':'candidatos_encontrados_no_equivalen_automaticamente_a_test_final',
                   'checkpoint_integrity':integrity.get('checkpoint',{}),
                   'training_index_integrity':integrity.get('training_index',{}),
                   'source_reports':source_audit,
                   'pairing_status':paired_status,
                   'coverage':coverage,
                   'recomputed':recomputed}

provenance={}
if PRED_PROVENANCE_JSON is not None:
    p=Path(PRED_PROVENANCE_JSON)
    if not p.is_file():raise FileNotFoundError('No existe PRED_PROVENANCE_JSON')
    data=json.loads(p.read_text(encoding='utf-8-sig'))
    provenance={'file':p.name,'sha256':sha256_file(p),
                'referenced_checkpoint_sha256':data.get('checkpoint_sha256',data.get('checkpointSha256')),
                'referenced_split_sha256':data.get('split_sha256',data.get('splitSha256')),
                'note':'revisar semántica con notebook fuente antes de atribuir.'}
source_provenance['prediction_provenance']=provenance

complete_pairing=paired_status=='pareado_completo' and coverage.get('match_test_patients_33') and coverage.get('match_test_slices_1218')
checkpoints_match=integrity.get('checkpoint',{}).get('status')=='match'
provenance_match=provenance.get('referenced_checkpoint_sha256')==EXPECTED['checkpoint_sha256']
# Aun con hashes, no se confirma metodología sin cotejo con notebook original.
if complete_pairing and checkpoints_match and provenance_match:
    verdict='RECALCULADO_CON_PROCEDENCIA_PARCIAL: requiere cotejar cálculo con notebook 45 y split original'
elif recomputed:
    verdict='EXPLORATORIO_NO_ATRIBUIBLE_AL_REPORTE_OFICIAL: falta vincular fuentes/metodología'
else:
    verdict='NO_RECALCULADO: hacen falta reporte original por clase o arrays de test vinculadas al checkpoint'
source_provenance['verdict']=verdict
source_provenance['interpretation_limits']=[
  'Dice/IoU original agregado no permite inferir por clase de manera unica.',
  'No se utiliza la partición oficial oculta del benchmark SPIDER; se trata del test propio 33 pacientes.',
  'Predicciones archivadas sólo sirven si corresponden al checkpoint final y a ese mismo split.',
  'Cambios en el promediado (pixels/slices/pacientes) pueden producir resultados distintos.',
  'Este notebook no realiza inferencia y no acredita validez clínica.',
]
OUT_DIR.mkdir(exist_ok=True,parents=True)
(OUT_DIR/'H04_resumen_anonimo.json').write_text(json.dumps(source_provenance,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
lines=['# H04 — auditoría sagital Dice/IoU por clase','','**Resultado:** '+verdict,'',
       '## Proveniencia esperada','- Checkpoint SHA-256: `'+EXPECTED['checkpoint_sha256']+'`',
       '- Índice de entrenamiento SHA-256: `'+EXPECTED['training_index_sha256']+'`',
       '- Test propio declarado: 33 pacientes, 1.218 cortes.',
       '- Dice macro test no-background original: 0,8934316063846954.',
       '- IoU macro test no-background original: 0,8079981902423006.','',
       '## Cobertura y control','- Pareamiento: '+paired_status,
       '- Hash checkpoint: '+integrity.get('checkpoint',{}).get('status','no_proporcionado'),
       '- Hash índice: '+integrity.get('training_index',{}).get('status','no_proporcionado'),
       '- Reportes JSON relevantes revisados: '+str(sum(r['status']=='legible' for r in source_audit)),'']
if recomputed:
    lines += ['## Métricas recalculadas (exploratorias hasta corroborar metodología)','',
             '| Clase | Dice global pixels | Dice media cortes | Dice media pacientes | IoU global pixels |','|---|---:|---:|---:|---:|']
    for c in recomputed['classes']:
        fmt=lambda v:'N/D' if v is None else f'{v:.6f}'
        lines.append(f"| {c['name']} | {fmt(c['dice_global_pixels'])} | {fmt(c['dice_mean_slice'])} | {fmt(c['dice_mean_patient'])} | {fmt(c['iou_global_pixels'])} |")
lines+=['','## Límites','- Métricas históricas no se modifican sin verificación del procedimiento fuente.',
         '- No implica evaluación clínica ni externa.',
         '- No publicar identificadores o arrays clínicos en GitHub.']
(OUT_DIR/'H04_informe_anonimo.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
zip_path=OUT_DIR/'H04_auditoria_sagital_por_clase_anonimizada.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ('H04_resumen_anonimo.json','H04_informe_anonimo.md'):
        archive.write(OUT_DIR/name,arcname=name)
print('\n',verdict)
print('ZIP generado:',zip_path)
print('Tamaño (bytes):',zip_path.stat().st_size)
from google.colab import files
files.download(str(zip_path))
